In [2]:
from pathlib import Path
from typing import Iterable, Optional

import glob, sys, os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

thismodule = sys.modules[__name__]


In [2]:
data_path = '/home/bfildier/analyses/FildierSaba2026/input/sensitivity_analysis'

In [3]:
! ls /home/bfildier/analyses/FildierSaba2026/input/sensitivity_analysis

delta_t_enriched.csv	 lambda_min_enriched.csv  Tbmin_enriched.csv
lambda_max_enriched.csv  sigma_enriched.csv


In [4]:
varids = 'Tbmin', 'lambda_min', 'sigma', 'lambda_max', 'delta_t'

for varid in varids:

    file_name = '%s_enriched.csv'%varid
    setattr(thismodule,'data_%s'%varid,pd.read_csv(os.path.join(data_path,file_name)))

In [5]:
data_Tbmin

,case_id,sigma_km,lambda_min_km,lambda_max_km,Tb_seed_K,Tb_anvil_K,delta_t_h,v_ref_kmh,n_cc,n_cc_final,...,final_Amax_homogeneity,fraction_after_initialization,IH25_75,Amax_homogeneity_25_75,final_IH25_75,final_Amax_homogeneity_25_75,INIT_IH25_75,INIT_Amax_homogeneity_25_75,initial_IH25_75,initial_Amax_homogeneity_25_75
0,C1,20.0,100.0,1500.0,210.0,235.0,3.0,15.0,38,38,...,0.995948,0.736842,0.987197,0.987197,0.987197,0.987197,0.512878,0.512878,0.512878,0.512878
1,C1,20.0,100.0,1500.0,211.0,235.0,3.0,15.0,42,42,...,0.996960,0.761905,0.982089,0.982089,0.982089,0.982089,0.514035,0.514035,0.514035,0.514035
2,C1,20.0,100.0,1500.0,212.0,235.0,3.0,15.0,41,41,...,0.997085,0.707317,0.978467,0.978467,0.978467,0.978467,0.667991,0.667991,0.667991,0.667991
3,C1,20.0,100.0,1500.0,213.0,235.0,3.0,15.0,41,41,...,0.995602,0.707317,0.983444,0.983444,0.983444,0.983444,0.614565,0.614565,0.614565,0.614565
4,C1,20.0,100.0,1500.0,214.0,235.0,3.0,15.0,41,41,...,0.996904,0.658537,0.982685,0.982685,0.982685,0.982685,0.600640,0.600640,0.600640,0.600640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,C9,20.0,100.0,1500.0,221.0,235.0,3.0,15.0,10,10,...,0.844118,0.100000,0.640691,0.640691,0.640691,0.640691,0.726446,0.726446,0.726446,0.726446
412,C9,20.0,100.0,1500.0,222.0,235.0,3.0,15.0,11,11,...,0.981691,0.181818,0.655467,0.655467,0.655467,0.655467,0.530497,0.530497,0.530497,0.530497
413,C9,20.0,100.0,1500.0,223.0,235.0,3.0,15.0,11,11,...,0.995923,0.272727,0.829788,0.829788,0.829788,0.829788,0.625085,0.625085,0.625085,0.625085
414,C9,20.0,100.0,1500.0,224.0,235.0,3.0,15.0,11,11,...,0.995923,0.272727,0.829892,0.829892,0.829892,0.829892,0.622516,0.622516,0.622516,0.622516


In [1]:
# Liste des cas a tracer
CASE_IDS = [
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8",
    "C9", "C10", "C11", "C12", "C13", "C14", "C15", "C16",
    "C17", "C18", "C19",
    "C21", "C22", "C23", "C24", "C25", "C26", "C27",
]

In [ ]:
def _safe_div(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)

    out = np.full_like(num, np.nan, dtype=float)
    m = np.isfinite(num) & np.isfinite(den) & (den != 0.0)
    out[m] = num[m] / den[m]

    return out

def parameter_display_name(parameter_name: str) -> str:
    return PARAMETER_TITLES.get(parameter_name, parameter_name)


def _get_case_x_values(data: pd.DataFrame, x_parameter: str) -> np.ndarray:
    """
    Version robuste :
    on prend l'union de toutes les valeurs du parametre, pas seulement C1.
    """
    if x_parameter not in data.columns:
        raise KeyError(
            f"Colonne x absente: {x_parameter}. "
            f"Colonnes disponibles: {list(data.columns)}"
        )

    x = np.asarray(data[x_parameter].dropna().unique(), dtype=float)
    return np.sort(x)

def _aligned_case_values(
    data: pd.DataFrame,
    case_id: str,
    x_parameter: str,
    x_ref: np.ndarray,
    y_diagnostic: str,
    y_denominator: Optional[str] = None,
    y_denominator_mean: Optional[str] = None,
) -> np.ndarray:
    if "case_id" not in data.columns:
        raise KeyError("Colonne 'case_id' absente.")

    if x_parameter not in data.columns:
        raise KeyError(f"Colonne x absente: {x_parameter}")

    if y_diagnostic not in data.columns:
        raise KeyError(
            f"Colonne diagnostic absente: {y_diagnostic}\n"
            f"Colonnes disponibles: {list(data.columns)}"
        )

    if y_denominator is not None and y_denominator not in data.columns:
        raise KeyError(f"Colonne denominateur absente: {y_denominator}")

    if y_denominator_mean is not None and y_denominator_mean not in data.columns:
        raise KeyError(f"Colonne denominateur moyen absente: {y_denominator_mean}")

    c_data = data.loc[data["case_id"] == case_id].copy()

    if c_data.empty:
        return np.full(x_ref.size, np.nan, dtype=float)

    c_data = c_data.sort_values(x_parameter)

    y = np.asarray(c_data[y_diagnostic], dtype=float)

    if y_denominator is not None:
        den = np.asarray(c_data[y_denominator], dtype=float)

        if y_denominator_mean is not None:
            den = den + 2.0 * np.asarray(c_data[y_denominator_mean], dtype=float)

        y = _safe_div(y, den)

    tmp = pd.DataFrame(
        {
            "x": np.asarray(c_data[x_parameter], dtype=float),
            "y": y,
        }
    )

    tmp = tmp.groupby("x", as_index=True)["y"].mean()

    y_aligned = tmp.reindex(x_ref).values.astype(float)

    return y_aligned

def _nanpercentile_axis0(y_all: np.ndarray, q: float) -> np.ndarray:
    """
    np.nanpercentile peut emettre des warnings si une colonne est full-NaN.
    Cette version retourne NaN proprement pour ces colonnes.
    """
    y_all = np.asarray(y_all, dtype=float)
    out = np.full(y_all.shape[1], np.nan, dtype=float)

    for j in range(y_all.shape[1]):
        col = y_all[:, j]
        col = col[np.isfinite(col)]

        if col.size > 0:
            out[j] = float(np.percentile(col, q))

    return out

def _plot_median_iqr_curve(
    ax,
    data: pd.DataFrame,
    *,
    x_parameter: str,
    metric: str,
    label: str,
    color,
    linestyle: str = "-",
    linewidth: float = 2.0,
    fill_alpha: float = 0.12,
    show_iqr: bool = True,
) -> bool:
    """
    Trace mediane inter-cas + IQR pour une métrique.
    Retourne False si la colonne est absente ou full-NaN.
    """
    if metric not in data.columns:
        return False

    x = _get_case_x_values(data, x_parameter)
    n_param = len(x)

    y_all = np.full((N_C, n_param), np.nan, dtype=float)

    for i_c, c_name in enumerate(CASE_IDS):
        y_all[i_c, :] = _aligned_case_values(
            data,
            c_name,
            x_parameter,
            x,
            metric,
        )

    if not np.isfinite(y_all).any():
        return False

    y_25 = _nanpercentile_axis0(y_all, 25)
    y_50 = _nanpercentile_axis0(y_all, 50)
    y_75 = _nanpercentile_axis0(y_all, 75)

    if show_iqr:
        ax.fill_between(
            x,
            y_25,
            y_75,
            color=color,
            alpha=fill_alpha,
            edgecolor=None,
        )

    ax.plot(
        x,
        y_50,
        color=color,
        linestyle=linestyle,
        linewidth=linewidth,
        label=label,
    )

    return True


def _style_axis(ax, *, xlabel: str, ylabel: str, title: str):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.25)

def subplotAllData(
    ax,
    data: pd.DataFrame,
    x_parameter: str,
    xlabel: str,
    y_diagnostic: str,
    ylabel: str,
    y_denominator: Optional[str] = None,
    y_denominator_mean: Optional[str] = None,
    col_diag="k",
    col_ylabel="k",
    one_minus: bool = False,
    exp_transform: bool = False,
    yscale: str = "linear",
    show_all_cases: bool = True,
    **kwargs,
):
    x = _get_case_x_values(data, x_parameter)
    n_param = len(x)

    y_all = np.full((N_C, n_param), np.nan, dtype=float)

    for i_c, c_name in enumerate(CASE_IDS):
        y = _aligned_case_values(
            data,
            c_name,
            x_parameter,
            x,
            y_diagnostic,
            y_denominator=y_denominator,
            y_denominator_mean=y_denominator_mean,
        )

        if one_minus:
            y = 1.0 - y
        elif exp_transform:
            y = 1.0 - np.exp(1.0 - y)

        y_all[i_c, :] = y

        if show_all_cases and np.isfinite(y).any():
            ax.plot(
                x,
                y,
                c=CMAP(NORM(i_c)),
                linewidth=1,
                alpha=0.5,
                **kwargs,
            )

    y_25 = _nanpercentile_axis0(y_all, 25)
    y_50 = _nanpercentile_axis0(y_all, 50)
    y_75 = _nanpercentile_axis0(y_all, 75)

    ax.fill_between(x, y_25, y_75, color=col_diag, alpha=0.5, edgecolor=None)
    ax.plot(x, y_50, color=col_diag, linewidth=2, alpha=1, **kwargs)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel, c=col_ylabel)
    ax.set_yscale(yscale)

def _plot_one_parameter_row_for_metric_pair(
    axs_row,
    data: pd.DataFrame,
    *,
    x_parameter: str,
    xlabel: str,
    default_x: float,
    parameter_name: str,
    ia_metric: str,
    ie_metric: str,
    legacy_heterogeneity: bool,
    n_ylim=(0, 200),
    amax_ylim=(0, 1e6),
    metric_ylim=(-0.01, 1.01),
) -> bool:
    """
    Trace une ligne avec 4 colonnes :

      1. Nombre final + fraction 1 - n_i/n_f
      2. Amax90 + max(Amax) + fraction enveloppe convexe
      3. IH10/90 + IH25/75 + Gini
      4. IA/IE final + IA/IE initial si disponibles
    """

    required_cols = [
        x_parameter,
        "n_cc",
        ia_metric,
        ie_metric,
    ]

    missing = [c for c in required_cols if c not in data.columns]

    if missing:
        for ax in axs_row:
            ax.axis("off")

        axs_row[0].text(
            0.5,
            0.5,
            f"{parameter_name}\ncolonnes absentes :\n{missing}",
            ha="center",
            va="center",
            fontsize=9,
        )

        print(
            f"[SKIP] {parameter_name} {ia_metric}/{ie_metric}: "
            f"colonnes absentes: {missing}"
        )

        return False

    param_title = parameter_display_name(parameter_name)

    # ==================================================================
    # Colonne 1 : nombre + fraction 1 - n_i/n_f
    # ==================================================================
    ax = axs_row[0]
    ax.axvline(x=default_x, linewidth=1, linestyle=":", c="k")

    _plot_median_iqr_curve(
        ax,
        data,
        x_parameter=x_parameter,
        metric="n_cc",
        label=r"$n_f$",
        color="black",
        linestyle="-",
        show_iqr=True,
    )

    _style_axis(
        ax,
        xlabel=xlabel,
        ylabel="Number of final aggregates",
        title=rf"{param_title} : number / fraction",
    )

    if n_ylim is not None:
        ax.set_ylim(n_ylim)

    ax2 = ax.twinx()

    plotted_frac = _plot_median_iqr_curve(
        ax2,
        data,
        x_parameter=x_parameter,
        metric="fraction_after_initialization",
        label=r"$1 - n_i/n_f$",
        color="grey",
        linestyle="--",
        show_iqr=True,
    )

    if plotted_frac:
        ax2.set_ylabel("fraction of aggregates\nforming after initialization", color="grey")
        ax2.tick_params(axis="y", colors="grey")
        ax2.set_ylim((-0.01, 1.01))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    if lines1 or lines2:
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="best")

    # ==================================================================
    # Colonne 2 : Amax90 + max(Amax) + enveloppe convexe normalisee
    # ==================================================================
    ax = axs_row[1]
    ax.axvline(x=default_x, linewidth=1, linestyle=":", c="k")

    _plot_median_iqr_curve(
        ax,
        data,
        x_parameter=x_parameter,
        metric="Amax_p90_km2",
        label=r"$A_{\max}^{90}$",
        color="brown",
        linestyle="-",
        show_iqr=True,
    )

    _plot_median_iqr_curve(
        ax,
        data,
        x_parameter=x_parameter,
        metric="Amax_max_km2",
        label=r"$\max(A_{\max})$",
        color="saddlebrown",
        linestyle="--",
        show_iqr=True,
    )

    _style_axis(
        ax,
        xlabel=xlabel,
        ylabel=r"Aggregate size $A_{\max}$" "\n" r"(km$^2$) in final segmentation",
        title=rf"{param_title} : size / envelope",
    )

    if amax_ylim is not None:
        ax.set_ylim(amax_ylim)
    ax.set_ylabel(r"Aggregate size $A_{\max}$" "\n" r"(km$^2$) in final segmentation", color="brown")
    ax.tick_params(axis="y", colors="brown")

    ax2 = ax.twinx()

    plotted_hull = _plot_median_iqr_curve(
        ax2,
        data,
        x_parameter=x_parameter,
        metric="convex_hull_lambda_fraction",
        label=r"$\max_{t,a} A_{\mathrm{hull}} / (\pi \lambda_{\max}^2)$",
        color="purple",
        linestyle="-.",
        show_iqr=True,
    )

    if plotted_hull:
        ax2.set_ylabel(
            r"Convex-hull fraction",
            color="purple",
        )
        ax2.tick_params(axis="y", colors="purple")
        ax2.set_ylim((-0.01, 1.05))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    if lines1 or lines2:
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="best")

    # ==================================================================
    # Colonne 3 : IH10/90 + IH25/75 + Gini
    # ==================================================================
    ax = axs_row[2]
    ax.axvline(x=default_x, linewidth=1, linestyle=":", c="k")

    # Axe gauche : hétérogénéités IH
    _plot_median_iqr_curve(
        ax,
        data,
        x_parameter=x_parameter,
        metric="IH",
        label=r"$I_H^{10/90}=1-P10/P90$",
        color="orange",
        linestyle="-",
        show_iqr=True,
    )

    _plot_median_iqr_curve(
        ax,
        data,
        x_parameter=x_parameter,
        metric="IH25_75",
        label=r"$I_H^{25/75}=1-P25/P75$",
        color="red",
        linestyle="--",
        show_iqr=True,
    )

    _style_axis(
        ax,
        xlabel=xlabel,
        ylabel=r"Heterogeneity $I_H$",
        title=rf"{param_title} : heterogeneity / Gini",
    )

    ax.set_ylim((-0.01, 1.01))
    ax.set_ylabel(r"Heterogeneity $I_H$", color="orange")
    ax.tick_params(axis="y", colors="orange")
    ax.spines["left"].set_color("orange")

    # Axe droit : Gini(Amax)
    ax_gini = ax.twinx()

    gini_color = "navy"
    _plot_median_iqr_curve(
        ax_gini,
        data,
        x_parameter=x_parameter,
        metric="Amax_gini",
        label=r"$Gini(A_{\max})$",
        color=gini_color,
        linestyle="-.",
        show_iqr=True,
    )

    ax_gini.set_ylabel(r"$Gini(A_{\max})$", color=gini_color)
    ax_gini.tick_params(axis="y", colors=gini_color)
    ax_gini.spines["right"].set_color(gini_color)
    ax_gini.set_ylim((-0.01, 1.01))

    # Légende combinée gauche + droite
    lines_left, labels_left = ax.get_legend_handles_labels()
    lines_right, labels_right = ax_gini.get_legend_handles_labels()

    ax.legend(
        lines_left + lines_right,
        labels_left + labels_right,
        fontsize=7,
        loc="best",
    )
    # ==================================================================
    # Colonne 4 : IA/IE final + IA/IE initial
    # ==================================================================
    ax = axs_row[3]
    ax.axvline(x=default_x, linewidth=1, linestyle=":", c="k")

    suffix = metric_suffix_from_name(ia_metric)

    init_ia_metric = f"INIT_IA{suffix}"
    init_ie_metric = f"INIT_IE{suffix}"

    _plot_median_iqr_curve(
        ax,
        data,
        x_parameter=x_parameter,
        metric=ia_metric,
        label=rf"{ia_metric} final",
        color="blue",
        linestyle="-",
        show_iqr=True,
    )

    _plot_median_iqr_curve(
        ax,
        data,
        x_parameter=x_parameter,
        metric=init_ia_metric,
        label=rf"{init_ia_metric} initial",
        color="royalblue",
        linestyle="--",
        show_iqr=True,
    )

    _style_axis(
        ax,
        xlabel=xlabel,
        ylabel=metric_axis_label(ia_metric, language="en", initial=False),
        title=rf"{param_title} : final / initial metrics",
    )

    ax2 = ax.twinx()

    _plot_median_iqr_curve(
        ax2,
        data,
        x_parameter=x_parameter,
        metric=ie_metric,
        label=rf"{ie_metric} final",
        color="green",
        linestyle="-",
        show_iqr=True,
    )

    _plot_median_iqr_curve(
        ax2,
        data,
        x_parameter=x_parameter,
        metric=init_ie_metric,
        label=rf"{init_ie_metric} initial",
        color="limegreen",
        linestyle="--",
        show_iqr=True,
    )

    ax.set_ylabel(metric_axis_label(ia_metric, language="en", initial=False), color="blue")
    ax.tick_params(axis="y", colors="blue")
    ax2.set_ylabel(metric_axis_label(ie_metric, language="en", initial=False), color="green")
    ax2.tick_params(axis="y", colors="green")

    if metric_ylim is not None:
        ax.set_ylim(metric_ylim)
        ax2.set_ylim(metric_ylim)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    if lines1 or lines2:
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="best")

    return True

In [ ]:
def plot_focus_grouped_by_metric_pair(
    *,
    ia_metric: str,
    ie_metric: str,
    data_sigma: pd.DataFrame,
    data_lambda_min: pd.DataFrame,
    data_Tbmin: pd.DataFrame,
    data_lambda_max: pd.DataFrame,
    data_delta_t: pd.DataFrame,
    out_dir: Path,
    formats: Iterable[str],
    legacy_heterogeneity: bool,
):
    """
    Produit une seule figure pour une paire IA/IE.

    Lignes :
      sigma
      lambda_min
      Tbmin
      lambda_max
      delta_t

    Colonnes :
      1. Nombre + fraction 1 - n_i/n_f
      2. Amax90 + max(Amax) + enveloppe convexe normalisee
      3. IH10/90 + IH25/75 + Gini
      4. IA/IE final + initial
    """

    suffix = metric_suffix_from_name(ia_metric)
    is_raw_cumulative = _is_raw_cumulative_suffix(suffix)

    metric_ylim = None if is_raw_cumulative else (-0.01, 1.01)

    parameter_specs = [
        dict(
            parameter_name="sigma",
            data=data_sigma,
            x_parameter="sigma_km",
            xlabel=r"$\sigma$ (km)",
            default_x=30,
            n_ylim=(0, 200),
            amax_ylim=(0, 1e6),
            metric_ylim=metric_ylim,
        ),
        dict(
            parameter_name="lambda_min",
            data=data_lambda_min,
            x_parameter="lambda_min_km",
            xlabel=r"$\lambda_{\min}$ (km)",
            default_x=100,
            n_ylim=(0, 500),
            amax_ylim=(0, 1e6),
            metric_ylim=metric_ylim,
        ),
        dict(
            parameter_name="Tbmin",
            data=data_Tbmin,
            x_parameter="Tb_seed_K",
            xlabel=r"$T_{b,\min}$ (K)",
            default_x=220,
            n_ylim=(0, 200),
            amax_ylim=(0, 1e6),
            metric_ylim=metric_ylim,
        ),
        # dict(
        #     parameter_name="lambda_max",
        #     data=data_lambda_max,
        #     x_parameter="lambda_max_km",
        #     xlabel=r"$\lambda_{\max}$ (km)",
        #     default_x=1500,
        #     n_ylim=(0, 200),
        #     amax_ylim=(0, 1e6),
        #     metric_ylim=metric_ylim,
        # ),
        # dict(
        #     parameter_name="delta_t",
        #     data=data_delta_t,
        #     x_parameter="delta_t_requested_h",
        #     xlabel=r"$\Delta t$ (h)",
        #     default_x=3,
        #     n_ylim=(0, 200),
        #     amax_ylim=(0, 1e6),
        #     metric_ylim=metric_ylim,
        # ),
    ]

    n_rows = len(parameter_specs)

    fig, axs = plt.subplots(
        n_rows,
        4,
        figsize=(18.5, 3.2 * n_rows),
        squeeze=False,
    )

    n_ok = 0

    for i, spec in enumerate(parameter_specs):
        ok = _plot_one_parameter_row_for_metric_pair(
            axs[i, :],
            spec["data"],
            x_parameter=spec["x_parameter"],
            xlabel=spec["xlabel"],
            default_x=spec["default_x"],
            parameter_name=spec["parameter_name"],
            ia_metric=ia_metric,
            ie_metric=ie_metric,
            legacy_heterogeneity=legacy_heterogeneity,
            n_ylim=spec["n_ylim"],
            amax_ylim=spec["amax_ylim"],
            metric_ylim=spec["metric_ylim"],
        )

        if ok:
            n_ok += 1

    fig.suptitle(
        focus_title_for_pair("all_parameters", ia_metric, ie_metric)
        + "\n"
        + rf"{ia_metric} / {ie_metric}",
        fontsize=14,
        y=0.995,
    )

    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.965])

    grouped_out_dir = out_dir / "by_metric"
    grouped_out_dir.mkdir(parents=True, exist_ok=True)

    stem = f"fig_by_metric_{ia_metric}_{ie_metric}_all_parameters"

    if n_ok == 0:
        print(f"[SKIP] figure {ia_metric}/{ie_metric}: aucune ligne traçable.")
        plt.close(fig)
        return

    save_figure(fig, grouped_out_dir, stem, formats)